[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab1_JSON_Mode_Pydantic.ipynb)

# M03 Lab 1 — JSON Mode & Pydantic Validation

**DADS 5250: Generative AI in Practice** | Northeastern University

Difficulty: ⭐⭐ | Time: ~10 min

## Learning Objectives

1. Force structured JSON responses using JSON mode
2. Extract structured data from unstructured text
3. Validate LLM outputs with Pydantic models

In [ ]:
!pip install -q openai tiktoken pydantic
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
import json
from pydantic import BaseModel
from openai import OpenAI
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

client = setup_openai()

---
## 1. JSON Mode

JSON mode forces the model to return **valid JSON**. Enable with `response_format={"type": "json_object"}` and instruct the model to respond in JSON.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Always respond in JSON format."},
        {"role": "user", "content": "List 3 popular programming languages with their primary use case."}
    ],
    response_format={"type": "json_object"},
)

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2))

---
## 2. Structured Data Extraction

Extract structured information from unstructured text — one of the most common real-world LLM use cases.

In [ ]:
invoice_text = """
Invoice #INV-2024-0847
Date: March 15, 2024
Bill To: Acme Corp, 123 Main St, Boston MA 02101

Items:
- Cloud hosting (monthly): $450.00
- API usage (10K calls): $125.50
- Support plan (premium): $200.00

Subtotal: $775.50
Tax (6.25%): $48.47
Total Due: $823.97
Payment Terms: Net 30
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Extract structured data from the invoice. Respond in JSON with keys: invoice_number, date, customer, items (list with name and amount), subtotal, tax, total."},
        {"role": "user", "content": invoice_text}
    ],
    response_format={"type": "json_object"},
)

invoice_data = json.loads(response.choices[0].message.content)
print(json.dumps(invoice_data, indent=2))

---
## 3. Pydantic Validation

[Pydantic](https://docs.pydantic.dev/) validates that the LLM's JSON matches your expected schema — catches malformed responses before they break your app.

In [ ]:
class InvoiceItem(BaseModel):
    name: str
    amount: float

class Invoice(BaseModel):
    invoice_number: str
    date: str
    customer: str
    items: list[InvoiceItem]
    subtotal: float
    tax: float
    total: float

try:
    validated = Invoice(**invoice_data)
    print(f"Invoice {validated.invoice_number}")
    print(f"Customer: {validated.customer}")
    for item in validated.items:
        print(f"  - {item.name}: ${item.amount:.2f}")
    print(f"Total: ${validated.total:.2f}")
    show_info("Pydantic validation passed!")
except Exception as e:
    print(f"Validation error: {e}")

---
## 4. Knowledge Check

In [ ]:
quiz([
    {
        "q": "What is the key difference between JSON mode and function calling?",
        "options": [
            "JSON mode is faster than function calling",
            "JSON mode returns free-form JSON; function calling returns structured calls matching a predefined schema",
            "Function calling only works with GPT-4",
            "There is no practical difference"
        ],
        "answer": 1,
        "explanation": "JSON mode forces valid JSON but the structure is flexible. Function calling defines exact schemas the model must follow."
    }
])

---
## 5. Hands-on: Multi-Document Extraction (Observational)

Run JSON extraction on 3 different document types and compare quality.

In [ ]:
documents = {
    "Restaurant Menu": """
    BELLA'S ITALIAN - LUNCH MENU
    Margherita Pizza - Fresh mozzarella, basil, tomato sauce - $14.95
    Caesar Salad - Romaine, parmesan, croutons, house dressing - $11.50
    Penne Arrabiata - Spicy tomato sauce, garlic, chili flakes - $16.00
    Tiramisu - Classic Italian dessert - $8.50
    """,
    "Job Posting": """
    SENIOR ML ENGINEER - TechCorp (Remote)
    Salary: $150,000 - $200,000
    Requirements: 5+ years Python, PyTorch/TensorFlow, cloud deployment (AWS/GCP)
    Nice to have: LLM fine-tuning experience, Kubernetes
    Benefits: Health, dental, 401k match, unlimited PTO
    Apply by: April 30, 2024
    """,
    "Product Review": """
    Samsung Galaxy S24 Ultra Review - 4.5/5 stars
    Pros: Amazing camera (200MP), great battery life (2 days), S-Pen is useful
    Cons: Expensive ($1,299), heavy (233g), AI features feel gimmicky
    Verdict: Best Android phone if budget isn't a concern.
    """
}

for doc_type, text in documents.items():
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Extract all structured data from this document. Return well-organized JSON."},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
    )
    print(f"--- {doc_type} ---")
    print(json.dumps(json.loads(r.choices[0].message.content), indent=2))
    print()

**Your Observations:** (double-click to edit)

| Document Type | Extraction Quality | Missing/Wrong Fields? | Notes |
|--------------|-------------------|----------------------|-------|
| Restaurant Menu | | | |
| Job Posting | | | |
| Product Review | | | |

---
## Summary

- **JSON mode**: `response_format={"type": "json_object"}` forces valid JSON
- **Pydantic**: Validates LLM outputs match your expected schema
- **Extraction**: Works well across document types — quality depends on prompt clarity

**Next:** M03 Lab 2 — Function Calling & Tool Use